In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("table_name", "hospitals", "Table Name")
dbutils.widgets.text("landing_timestamp", "2026-10-19_10:09:29", "Landing Timestamp")
dbutils.widgets.text("source_system", "", "neon_postgres")
dbutils.widgets.text("source_file_format", "", "neon_postgres")

table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_system = dbutils.widgets.get("source_system")  
source_file_format = dbutils.widgets.get("source_file_format")  

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(f"/Volumes/dbw_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")
else:
     landing_df = spark.read.format("parquet").load(f"/Volumes/dbw_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")

In [0]:
updated_df = landing_df.withColumn(
    "landing_timestamp", lit(landing_timestamp)
).withColumn("insert_timestamp", current_timestamp())

In [0]:
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"dbw_caresync.bronze.{table_name}")

In [0]:
%sql

select * from dbw_caresync.silver.hospitals